# Focused BSk24 dataset pilot

A thin front end to the same public scientific workflow. Produces one flat `plots/` folder containing pressure–energy density, sound speed, M–R, k2–M and Lambda–M for all accepted EoSs in this experiment, plus persistent H labels and full primary tables.

**Default: `dataset_40`, up to 6 case workers.** One 40-point stellar stage at rtol=1e-10 and atol=1e-12, with tides at every point, fixed-mass solves and unchanged maximum-mass refinement. STRICT thermodynamics, raw certification and surface/radial conventions are retained. This experimental single-stage profile has no per-case stellar refinement envelope and is not full STRICT certification. Sparse curves and difficult turning points require benchmark review before ML use. Small batches or lower-core machines use fewer workers; nested pools remain disabled.

Explicit alternatives: `dataset` uses 61 points and `dataset_20` uses 20 points at rtol=1e-10/atol=1e-12. `dataset_10_tighter` uses 10 points at rtol=1e-11/atol=1e-13, changing both sampling and tolerances. `dataset_relaxed` uses 61 points at rtol=1e-8/atol=1e-10; `dataset_relaxed_80` uses 80 points at those relaxed tolerances. All retain all-node tides and the other governed settings. None is STRICT certification.

Restart the kernel after package changes. First Run All with False; review counts and the new destination. Then change only the execution flag to True. Do not run a second heavy notebook concurrently.

In [ ]:
from eos_generation.notebook import NotebookSettings, get_notebook_session

notebook_session = get_notebook_session()

## Settings

All energy-density coordinates use MeV fm^-3. Lists form a Cartesian product. Start small. This focused notebook fixes stellar calculation, disables extended diagnostics, and uses the experimental dataset profile. Fixed 1.4-solar-mass results and maximum-mass availability remain separate. Accepted does not mean every observable is available.

In [ ]:


'''
# BIG POSITIVE-AMPLITUDE RUN

AMPLITUDES = [0.04, 0.08, 0.12, 0.16, 0.20, 0.24, 0.28, 0.32]
EPSILON_MATCH = 80.0

CENTER = [150.0, 225.0, 300.0, 375.0, 500.0]
WIDTH = [150.0, 300.0, 450.0, 600.0, 750.0]
RAMP_WIDTH = [175.0, 200.0, 225.0, 275.0, 350.0]

CALCULATION = "stellar"
FIXED_MASSES = [1.4]
PRECISION = "dataset_40"
DIAGNOSTICS = "off"

# First pass: preview only.
EXECUTE_REVIEWED_PLAN = False
'''

AMPLITUDES = [-0.04, -0.06, -0.08, -0.10, -0.12, -0.14,
              -0.16, -0.18, -0.20, -0.22, -0.24]
EPSILON_MATCH = 80.0

CENTER = [200.0, 450.0, 500.0, 700.0, 950.0]
WIDTH = [150.0, 500.0, 900.0]
RAMP_WIDTH = [175.0, 225.0, 250.0, 275.0, 300.0, 350.0]

CALCULATION = "stellar"
FIXED_MASSES = [1.4]
PRECISION = "dataset_40"
DIAGNOSTICS = "off"

EXECUTE_REVIEWED_PLAN = False

In [ ]:
if CALCULATION != "stellar" or PRECISION not in {"dataset", "dataset_10_tighter", "dataset_20", "dataset_40", "dataset_relaxed", "dataset_relaxed_80"} or DIAGNOSTICS != "off":
    raise ValueError("This notebook requires stellar calculation, a dataset-family precision, and diagnostics off.")

settings = NotebookSettings.from_values(
    amplitudes=AMPLITUDES,
    epsilon_match=EPSILON_MATCH,
    center=CENTER,
    width=WIDTH,
    ramp_width=RAMP_WIDTH,
    calculation=CALCULATION,
    fixed_masses=FIXED_MASSES,
    precision=PRECISION,
    diagnostics=DIAGNOSTICS,
)

In [ ]:
import hashlib

presentation_sources = {
    name: hashlib.sha256((notebook_session.repository_root / "notebooks" / name).read_bytes()).hexdigest()
    for name in ("eos_catalogue.py", "build_dataset_plots.py")
}
if not EXECUTE_REVIEWED_PLAN:
    reviewed_presentation_sources = presentation_sources
elif globals().get("reviewed_presentation_sources") != presentation_sources:
    raise RuntimeError("Presentation source changed or was not previewed; Run All with EXECUTE_REVIEWED_PLAN=False first.")

notebook_run = notebook_session.prepare(
    settings, record_preview=not EXECUTE_REVIEWED_PLAN
)
print(notebook_run.summary_text())

In [ ]:
experiment_result = notebook_session.execute(
    notebook_run, current_settings=settings, execute=EXECUTE_REVIEWED_PLAN,
)
if experiment_result is None:
    print("EXECUTE_REVIEWED_PLAN=False: no calculations or writes.")
else:
    import json
    import subprocess
    import sys
    from pathlib import Path
    from IPython.display import Markdown, display

    print(experiment_result.summary_text())
    if reviewed_presentation_sources != {
        name: hashlib.sha256((notebook_session.repository_root / "notebooks" / name).read_bytes()).hexdigest()
        for name in reviewed_presentation_sources
    }:
        raise RuntimeError("Scientific run complete but presentation source changed. Preserve packets; recover reporting separately.")
    root = notebook_session.repository_root
    eos_data = notebook_run.planning_root / "EOS_DATA"
    plots = notebook_run.planning_root / "plots"
    try:
        result = subprocess.run(
            [sys.executable, str(root / "notebooks/eos_catalogue.py"),
             "--repository-root", str(root), "--experiment", str(notebook_run.output_root),
             "--destination", str(eos_data)],
            cwd=root, check=True, capture_output=True, text=True,
        )
        catalogue_result = json.loads(result.stdout)
        result = subprocess.run(
            [sys.executable, str(root / "notebooks/build_dataset_plots.py"),
             "--repository-root", str(root), "--experiment", str(notebook_run.output_root),
             "--destination", str(plots), "--eos-data", str(eos_data)],
            cwd=root, check=True, capture_output=True, text=True,
        )
        plot_result = json.loads(result.stdout)
        print(f"Five combined plot families; {plot_result['unique_eos_count']} physical EoSs; plotting solver calls: 0.")
    except (subprocess.CalledProcessError, json.JSONDecodeError) as error:
        print("Scientific calculation is complete. Do not rerun it for a presentation failure.")
        print(getattr(error, "stderr", "") or str(error))
        raise
    locations = {
        "Five combined plots": plots,
        "Labelled primary tables": eos_data,
        "Friendly EoS catalogue": eos_data / "eos_catalogue.csv",
        "Canonical case mapping": eos_data / "case_aliases.csv",
        "Authoritative experiment": notebook_run.output_root,
        "Shared numbering — archive this": Path(catalogue_result["catalogue_path"]),
    }
    links = [f"- [{label}](../{path.relative_to(root).as_posix()})" for label, path in locations.items()]
    display(Markdown("## Dataset locations\n\n" + "\n".join(links)))

## Interpretation and next steps

The five figures use only validated saved data from this experiment. The sound-speed plot stops at each accepted causal endpoint; rejected raw profiles remain scientific evidence in the packet, not plotted as accepted EoSs. Stellar curves follow attempted-pressure order through the sampled peak, preserve failure gaps and require validated tides. That plotted prefix is not an independent radial-stability certificate or a resolved maximum mass. Read availability and convergence statuses.

The source packets preserve all mandatory certification and residual evidence even though extra figures are disabled. Nothing is clipped, extrapolated or repaired. H labels identify the effective deformed-BSk24 family, not microscopic composition. Keep unavailable values missing and split ML datasets by whole physical EoSs/geometries. This notebook does not yet create the professor's ten aligned ML tuples or certify a production accuracy budget.

Archive the append-only `runs/eos_catalogue/` registry with your data. Do not reset numbering. Read [dataset workflow and benchmark](../docs/dataset.md) before a large campaign.